# CRIVO — continuar o leitor já treinado (~17M)

Este caderno aproveita os pesos próprios existentes: pré-treino de 20.000 passos e checkpoint do leitor. Ajusta os dois últimos blocos, a normalização final e a cabeça para distinguir fatos parecidos e perguntas sem resposta.

Selecione **GPU T4 ou melhor** e execute as células na ordem. São **1.000 passos adicionais**, sem novo pré-treino nem download da Wikipédia. O tempo restante aparece após os primeiros 20 passos; as avaliações levam tempo adicional.

O treino compara o candidato com o leitor original no tutor. Assuntos reservados e conjuntos congelados ficam fora do treino. Nenhum peso recebe aprovação automática.


In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, sys, json, hashlib, shutil
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Selecione GPU em Ambiente de execução → Alterar o tipo de ambiente.')
drive.mount('/content/drive')
ROOT = Path('/content/CRIVO-ajuste-leitor')
REVISAO = '05a9fce4240d159813c0b138989d9ea4ff9fcb41'
DRIVE = Path('/content/drive/MyDrive/CRIVO/leitor-ajuste-v1')
DRIVE.mkdir(parents=True, exist_ok=True)
if not (ROOT / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/HROSONE/CRIVO.git', str(ROOT)], check=True)
subprocess.run(['git', 'diff', '--exit-code'], cwd=ROOT, check=True)
subprocess.run(['git', 'fetch', '--depth', '1', 'origin', REVISAO], cwd=ROOT, check=True)
subprocess.run(['git', 'checkout', '--detach', REVISAO], cwd=ROOT, check=True)
ORIGEM = ROOT / 'experimentos/pesos_base/leitor_transformer'
ORIGEM_SHA = '603ff3b6765d9fc82ff53517560ba3f1abca595e27f5a98c55911f1bb9a2b6dc'
assert hashlib.sha256((ORIGEM / 'pesos_numpy.npz').read_bytes()).hexdigest() == ORIGEM_SHA, 'Pesos diferentes do checkpoint preparado.'
print('GPU:', torch.cuda.get_device_name(0))
print('Pesos existentes conferidos. Não é treino do zero.')


In [ ]:
# Preserva o PyTorch com CUDA instalado no Colab.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy>=1.24,<3'], check=True)
SCRIPT = ROOT / 'scripts/continuar_leitor_colab.py'
SCRIPT.write_text('"""Ajuste contrastivo do leitor próprio já treinado; nunca pré-treina do zero."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport random\nimport shutil\nimport sys\nimport time\n\nimport numpy as np\nimport torch\nfrom torch import nn\n\n\ndef sha256(caminho):\n    return hashlib.sha256(Path(caminho).read_bytes()).hexdigest()\n\n\ndef carregar(origem, dispositivo):\n    from linguagem_profunda import Configuracao, LinguagemProfunda\n    from pontuador_frases import BPE\n    origem = Path(origem)\n    meta = json.loads((origem / \'meta.json\').read_text(encoding=\'utf-8\'))\n    with np.load(origem / \'pesos_numpy.npz\', allow_pickle=False) as z:\n        config = json.loads(str(z[\'meta\'].item()))\n        chaves = (\'vocabulario\', \'dimensao\', \'camadas\', \'cabecas\', \'contexto\', \'dropout\')\n        cfg = Configuracao(**{k: config[k] for k in chaves})\n        if config[\'tokenizer_sha256\'] != sha256(origem / \'tokenizer.json\'):\n            raise ValueError(\'Tokenizador não corresponde aos pesos existentes.\')\n\n        class Leitor(nn.Module):\n            def __init__(self):\n                super().__init__()\n                self.base = LinguagemProfunda(cfg)\n                self.cabeca = nn.Linear(cfg.dimensao, 1)\n\n            def forward(self, ids, ultimos):\n                b = self.base\n                x = b.embedding(ids) + b.posicao(torch.arange(ids.shape[1], device=ids.device))\n                for bloco in b.blocos:\n                    x = bloco(x)\n                x = b.norm(x)\n                return self.cabeca(x[torch.arange(len(ids), device=ids.device), ultimos]).squeeze(-1)\n\n        modelo = Leitor()\n        estado = {}\n        for nome, tensor in modelo.state_dict().items():\n            chave = nome[5:] if nome.startswith(\'base.\') else nome\n            if chave not in z or tuple(z[chave].shape) != tuple(tensor.shape):\n                raise ValueError(\'Peso ausente ou formato incompatível: \' + chave)\n            estado[nome] = torch.from_numpy(np.array(z[chave], dtype=np.float32, copy=True))\n        modelo.load_state_dict(estado, strict=True)\n    return modelo.to(dispositivo), BPE(origem / \'tokenizer.json\'), meta\n\n\ndef configurar_ajuste(modelo, camadas):\n    if not 1 <= camadas <= len(modelo.base.blocos):\n        raise ValueError(\'Número de camadas inválido.\')\n    for parametro in modelo.parameters():\n        parametro.requires_grad_(False)\n    for modulo in list(modelo.base.blocos[-camadas:]) + [modelo.base.norm, modelo.cabeca]:\n        for parametro in modulo.parameters():\n            parametro.requires_grad_(True)\n\n\ndef modo_treino(modelo, camadas):\n    modelo.train()\n    for bloco in modelo.base.blocos[:-camadas]:\n        bloco.eval()\n\n\ndef perda_grupo(logits, alvo):\n    # Fatos da MESMA ficha competem; zero representa "nenhum responde".\n    alvo_rank = len(logits) if alvo is None else alvo\n    ranking = torch.nn.functional.cross_entropy(\n        torch.cat([logits, logits.new_zeros(1)])[None],\n        torch.tensor([alvo_rank], device=logits.device))\n    y = torch.zeros_like(logits)\n    if alvo is not None:\n        y[alvo] = 1\n    binaria = torch.nn.functional.binary_cross_entropy_with_logits(\n        logits, y, pos_weight=logits.new_tensor(max(1, min(4, len(logits) - 1))))\n    return ranking + .5 * binaria\n\n\ndef atualizar(modelo, bpe, grupos, rng, opt, lote, camadas, dispositivo):\n    from scripts.treinar_leitor_transformer import lote_tensores\n    positivos, negativos = grupos\n    if not positivos or not negativos:\n        raise ValueError(\'Treino exige exemplos respondíveis e sem resposta.\')\n    amostra = [rng.choice(negativos if rng.random() < .25 else positivos) for _ in range(lote)]\n    pares, limites = [], []\n    for pergunta, fatos, alvo in amostra:\n        ini = len(pares)\n        pares.extend((pergunta, fato) for fato in fatos)\n        limites.append((ini, len(pares), alvo))\n    ids, ultimos = lote_tensores(pares, bpe, modelo.base.config.contexto, dispositivo)\n    modo_treino(modelo, camadas)\n    opt.zero_grad(set_to_none=True)\n    logits = modelo(ids, ultimos)\n    perda = torch.stack([perda_grupo(logits[a:b], alvo) for a, b, alvo in limites]).mean()\n    if not torch.isfinite(perda):\n        raise RuntimeError(\'Perda não finita; treino interrompido.\')\n    perda.backward()\n    torch.nn.utils.clip_grad_norm_([p for p in modelo.parameters() if p.requires_grad], 1.)\n    opt.step()\n    return float(perda.detach())\n\n\ndef gravar_checkpoint(caminho, modelo, opt, rng, passo, assinatura, melhor):\n    caminho = Path(caminho)\n    caminho.parent.mkdir(parents=True, exist_ok=True)\n    estado = {\'modelo\': modelo.state_dict(), \'otimizador\': opt.state_dict(),\n              \'rng_python\': rng.getstate(), \'rng_torch\': torch.get_rng_state(),\n              \'rng_cuda\': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],\n              \'passo\': passo, \'assinatura\': assinatura, \'melhor\': melhor}\n    temporario = caminho.with_suffix(\'.tmp\')\n    torch.save(estado, temporario)\n    temporario.replace(caminho)\n\n\ndef retomar(caminho, modelo, opt, rng, assinatura):\n    estado = torch.load(caminho, map_location=\'cpu\', weights_only=True)\n    if estado[\'assinatura\'] != assinatura:\n        raise ValueError(\'Checkpoint de outro código, corpus, pesos ou configuração. Use outra pasta.\')\n    modelo.load_state_dict(estado[\'modelo\'], strict=True)\n    opt.load_state_dict(estado[\'otimizador\'])\n    rng.setstate(estado[\'rng_python\'])\n    torch.set_rng_state(estado[\'rng_torch\'])\n    if estado[\'rng_cuda\']:\n        if not torch.cuda.is_available():\n            raise ValueError(\'Checkpoint de GPU exige GPU para a continuação.\')\n        torch.cuda.set_rng_state_all(estado[\'rng_cuda\'])\n    return estado[\'passo\'], estado[\'melhor\']\n\n\ndef exportar(modelo, origem, saida, meta_original, passo, metricas, baseline):\n    saida = Path(saida)\n    saida.mkdir(parents=True, exist_ok=True)\n    shutil.copyfile(Path(origem) / \'tokenizer.json\', saida / \'tokenizer.json\')\n    config = vars(modelo.base.config).copy()\n    config.update(passo=passo, tokenizer_sha256=sha256(saida / \'tokenizer.json\'))\n    pesos = {k: v.detach().float().cpu().numpy().astype(np.float16)\n             for k, v in modelo.base.state_dict().items()}\n    pesos[\'cabeca.weight\'] = modelo.cabeca.weight.detach().float().cpu().numpy()\n    pesos[\'cabeca.bias\'] = modelo.cabeca.bias.detach().float().cpu().numpy()\n    np.savez_compressed(saida / \'pesos_numpy.npz\', meta=np.array(json.dumps(config)), **pesos)\n    meta = json.loads(json.dumps(meta_original))\n    meta[\'validacao_tutor\'] = metricas\n    meta[\'ajuste_contrastivo\'] = {\'passos_adicionais\': passo, \'baseline_tutor\': baseline,\n                                \'origem_sha256\': sha256(Path(origem) / \'pesos_numpy.npz\'),\n                                \'objetivo\': \'ranking de fatos da mesma ficha e nenhuma resposta\'}\n    meta[\'controle\'] = {\'aprovado\': False, \'criterio\': \'Requer ganho no controle e testes congelados.\'}\n    (saida / \'meta.json\').write_text(json.dumps(meta, ensure_ascii=False, indent=2) + \'\\n\', encoding=\'utf-8\')\n\n\ndef main():\n    ap = argparse.ArgumentParser(description=__doc__)\n    ap.add_argument(\'--repo\', required=True)\n    ap.add_argument(\'--origem\', required=True)\n    ap.add_argument(\'--saida\', required=True)\n    ap.add_argument(\'--checkpoint\', required=True)\n    ap.add_argument(\'--passos\', type=int, default=1000)\n    ap.add_argument(\'--lote\', type=int, default=4, help=\'Perguntas por passo, com todos os fatos de cada ficha\')\n    ap.add_argument(\'--camadas\', type=int, default=2)\n    ap.add_argument(\'--lr\', type=float, default=1e-5)\n    ap.add_argument(\'--avaliar-a-cada\', type=int, default=200)\n    ap.add_argument(\'--checkpoint-a-cada\', type=int, default=100)\n    ap.add_argument(\'--max-horas\', type=float, default=1.5)\n    ap.add_argument(\'--dispositivo\', default=\'cuda\')\n    args = ap.parse_args()\n    if min(args.passos, args.lote, args.avaliar_a_cada, args.checkpoint_a_cada) <= 0:\n        ap.error(\'Passos, lote e intervalos devem ser positivos.\')\n    if args.dispositivo == \'cuda\' and not torch.cuda.is_available():\n        ap.error(\'Selecione GPU no Colab.\')\n    sys.path.insert(0, str(Path(args.repo).resolve()))\n    sys.path.insert(0, str(Path(args.repo).resolve() / \'scripts\'))\n    from scripts.treinar_leitor_transformer import exemplos_sinteticos, exemplos_tutor, avaliar\n    from crivo import Crivo\n    torch.set_num_threads(2)\n    torch.manual_seed(20261008)\n    comp = Crivo().compositor\n    grupos = exemplos_sinteticos(comp, random.Random(20261008))\n    validacao = exemplos_tutor(comp)\n    # Usa os mesmos assuntos excluídos do treinador original; tutor/congelados nunca treinam.\n    modelo, bpe, meta_original = carregar(args.origem, args.dispositivo)\n    configurar_ajuste(modelo, args.camadas)\n    opt = torch.optim.AdamW([p for p in modelo.parameters() if p.requires_grad], lr=args.lr, weight_decay=.01)\n    rng = random.Random(20261008)\n    corpus_sha = hashlib.sha256(json.dumps([grupos, validacao], ensure_ascii=False).encode()).hexdigest()\n    assinatura = {\'origem\': sha256(Path(args.origem) / \'pesos_numpy.npz\'),\n                  \'tokenizador\': sha256(Path(args.origem) / \'tokenizer.json\'),\n                  \'codigo\': sha256(__file__), \'corpus\': corpus_sha,\n                  \'lr\': args.lr, \'lote\': args.lote, \'camadas\': args.camadas,\n                  \'dispositivo\': args.dispositivo, \'torch\': str(torch.__version__),\n                  \'modelo\': sha256(Path(args.repo) / \'linguagem_profunda.py\'),\n                  \'sequencia\': sha256(Path(args.repo) / \'leitor_transformer.py\'),\n                  \'treinador_base\': sha256(Path(args.repo) / \'scripts/treinar_leitor_transformer.py\')}\n    grupos = ([g for g in grupos if g[2] is not None], [g for g in grupos if g[2] is None])\n    saida = Path(args.saida)\n    baseline_path = saida / \'baseline.json\'\n    saida.mkdir(parents=True, exist_ok=True)\n    checkpoint = Path(args.checkpoint)\n    if checkpoint.exists():\n        if not baseline_path.exists() or not (saida / \'melhor\' / \'pesos_numpy.npz\').exists():\n            raise ValueError(\'Checkpoint sem baseline ou melhor candidato; restaure a pasta completa.\')\n        baseline = json.loads(baseline_path.read_text())\n        passo, melhor = retomar(checkpoint, modelo, opt, rng, assinatura)\n    else:\n        baseline = avaliar(modelo, bpe, modelo.base.config.contexto, validacao, args.dispositivo)\n        baseline_path.write_text(json.dumps(baseline, indent=2) + \'\\n\')\n        passo, melhor = 0, baseline\n        exportar(modelo, args.origem, saida / \'melhor\', meta_original, 0, baseline, baseline)\n    if passo > args.passos:\n        raise ValueError(\'O checkpoint já ultrapassou o horizonte solicitado.\')\n    print(\'Base existente:\', meta_original[\'base\'][\'passo_pretreino\'], \'passos de pré-treino; mantida.\', flush=True)\n    print(\'Baseline tutor:\', baseline, \'| retomando ajuste no passo\', passo, flush=True)\n    inicio, passo_inicial = time.monotonic(), passo\n    tempo_treino = 0.\n    while passo < args.passos:\n        inicio_passo = time.monotonic()\n        perda = atualizar(modelo, bpe, grupos, rng, opt, args.lote, args.camadas, args.dispositivo)\n        tempo_treino += time.monotonic() - inicio_passo\n        passo += 1\n        fim_tempo = args.max_horas > 0 and time.monotonic() - inicio >= args.max_horas * 3600\n        if passo % 20 == 0:\n            seg = tempo_treino / max(1, passo - passo_inicial)\n            print(\'passo %d/%d perda %.4f | %.2fs/passo | treino restante ~%.1fmin (avaliações à parte)\' %\n                  (passo, args.passos, perda, seg, (args.passos - passo) * seg / 60), flush=True)\n        if passo % args.avaliar_a_cada == 0 or passo == args.passos or fim_tempo:\n            metricas = avaliar(modelo, bpe, modelo.base.config.contexto, validacao, args.dispositivo)\n            print(\'Validação tutor:\', metricas, flush=True)\n            sem_regressao = all(metricas[k] >= baseline[k] for k in (\'acerto_fato\', \'auc_respondivel\'))\n            if sem_regressao and sum(metricas[k] for k in (\'acerto_fato\', \'auc_respondivel\')) > sum(\n                    melhor[k] for k in (\'acerto_fato\', \'auc_respondivel\')):\n                melhor = metricas\n                exportar(modelo, args.origem, saida / \'melhor\', meta_original, passo, metricas, baseline)\n        if passo % args.checkpoint_a_cada == 0 or passo == args.passos or fim_tempo:\n            gravar_checkpoint(checkpoint, modelo, opt, rng, passo, assinatura, melhor)\n            print(\'Checkpoint único gravado:\', passo, flush=True)\n        if fim_tempo:\n            print(\'Pausado por tempo; execute esta célula novamente para continuar.\', flush=True)\n            break\n    relatorio = {\'baseline\': baseline, \'melhor_tutor\': melhor, \'passos_adicionais\': passo,\n                 \'concluido\': passo == args.passos, \'aprovado\': False, \'assinatura\': assinatura}\n    (saida / \'relatorio.json\').write_text(json.dumps(relatorio, ensure_ascii=False, indent=2) + \'\\n\')\n    print(json.dumps(relatorio, ensure_ascii=False), flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n', encoding='utf-8')

def executar(cmd):
    with subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for linha in proc.stdout:
            print(linha, end='', flush=True)
    if proc.returncode:
        raise RuntimeError('O processo terminou com erro; veja o log acima.')


## Ajuste e retomada

Execute esta célula novamente após uma pausa. O checkpoint guarda modelo, otimizador e estados aleatórios; a continuação exige o mesmo código, dados e configuração. Cada sessão pausa após 1,5 hora. No Drive ficam um checkpoint sobrescrito, o melhor candidato e os relatórios. Os pesos originais do repositório são preservados.


In [ ]:
PASSOS = 1000
SAIDA = DRIVE / 'ajuste'
CHECKPOINT = DRIVE / 'checkpoint.pt'
executar([sys.executable, '-u', str(SCRIPT), '--repo', str(ROOT), '--origem', str(ORIGEM),
          '--saida', str(SAIDA), '--checkpoint', str(CHECKPOINT), '--passos', str(PASSOS),
          '--lote', '4', '--camadas', '2', '--lr', '1e-5', '--avaliar-a-cada', '200',
          '--checkpoint-a-cada', '100', '--max-horas', '1.5', '--dispositivo', 'cuda'])


## Resultado

A célula abaixo cria o pacote quando os 1.000 passos terminarem. Se nenhuma avaliação melhorar acerto e separação entre respondíveis e sem resposta, o pacote conserva os pesos do leitor original. Melhora no tutor ainda precisa ser confirmada no controle de leitura e nos conjuntos congelados antes da instalação.


In [ ]:
relatorio = json.loads((SAIDA / 'relatorio.json').read_text())
if not relatorio['concluido']:
    raise RuntimeError('O ajuste está pausado; retome a célula anterior antes de empacotar.')
assert hashlib.sha256((ORIGEM / 'pesos_numpy.npz').read_bytes()).hexdigest() == ORIGEM_SHA
print(json.dumps(relatorio, ensure_ascii=False, indent=2))
PACOTE = Path('/content/leitor-ajustado')
if PACOTE.exists():
    shutil.rmtree(PACOTE)
shutil.copytree(SAIDA / 'melhor', PACOTE / 'leitor_transformer')
shutil.copy(SAIDA / 'relatorio.json', PACOTE / 'relatorio_ajuste.json')
shutil.make_archive(str(DRIVE / 'resultado_leitor_ajustado'), 'zip', PACOTE)
print('Resultado:', DRIVE / 'resultado_leitor_ajustado.zip')
print('Aprovação pendente: enviar este ZIP para avaliar controle e testes congelados.')
